<a href="https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/milestone_3_baseline_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 3 — Heuristic baseline and model comparison

This stage tests whether a model can prioritize the current `trend_direction == 'down'` pages better than the existing stale high-traffic heuristic. It uses the CSV and holds out whole clients. It does not make future-warning claims: the snapshot does not verify when the feature values were available.

Stage 2 found similar average precision for complete-case, median, and median-plus-missing-flag approaches (about 0.61); median plus flags had the strongest mean Precision@50 (0.652) and lower split variation. This stage uses train-fold median imputation with missingness indicators for the model candidates. The heuristic remains unchanged: pages updated more than 180 days ago rank by `impressions_90d`.

In [1]:
# Open from the FlyRank repository, or clone it in Colab.
import os, sys, subprocess
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/flyrank-bih/flyrank-ml-internship-starter'
REPO_DIR = 'flyrank-ml-internship-starter'
CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if IN_COLAB and not CSV_PATH.exists():
    if not Path(REPO_DIR).is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
else:
    while not CSV_PATH.exists() and Path.cwd() != Path('/'):
        os.chdir('..')
        CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError('Expected data/raw/content_refresh_anonymized.csv in the FlyRank repo.')
print('CSV:', CSV_PATH.resolve())

CSV: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(CSV_PATH)
required = ['content_id','client_id','trend_direction','days_since_last_update',
            'content_age_days','impressions_90d','avg_position','ctr']
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f'Missing required fields: {missing}')
df['trend_direction'] = df['trend_direction'].astype('string').str.strip().str.lower()
df['is_declining_label'] = df['trend_direction'].map(
    {'down':1,'new':0,'up':0,'flat':0,'stable':0}
).astype('Int64')

FEATURES = ['days_since_last_update','content_age_days','impressions_90d','avg_position','ctr']
X = df[FEATURES].apply(pd.to_numeric, errors='coerce').copy()
X['avg_position'] = X['avg_position'].mask(X['avg_position'].eq(0))  # zero means unavailable
X['staleness_pos_interaction'] = X['days_since_last_update'] * X['avg_position']
FEATURES = list(X.columns)
valid = (df['is_declining_label'].notna() & df['content_id'].notna() &
         df['client_id'].notna() & ~df['content_id'].duplicated(keep=False))
X = X.loc[valid].reset_index(drop=True)
y = df.loc[valid,'is_declining_label'].astype(int).reset_index(drop=True)
groups = df.loc[valid,'client_id'].reset_index(drop=True)
eval_df = df.loc[valid].reset_index(drop=True)
print(f'Rows: {len(X):,}; clients: {groups.nunique()}; positive rate: {y.mean():.3f}')

Rows: 30,000; clients: 32; positive rate: 0.542


## Evaluation design

Every candidate is scored on the same five 75/25 `GroupShuffleSplit` partitions by `client_id`; an assertion checks that train and test clients do not overlap. The baseline is evaluated on the same held-out rows. We report average precision, ROC AUC, and Precision@20/50/100 because the system is intended to produce a review queue. The group-split results are exploratory with only 32 clients.

In [3]:
RANDOM_STATE = 42
N_SPLITS = 5
gss = GroupShuffleSplit(n_splits=N_SPLITS, test_size=0.25, random_state=RANDOM_STATE)

def precision_at_k(scores, labels, k):
    scores, labels = np.asarray(scores), np.asarray(labels)
    n = min(k, len(labels))
    return float(labels[np.argsort(-scores)[:n]].mean()) if n else np.nan

def metrics(y_true, scores):
    return {
        'average_precision': float(average_precision_score(y_true, scores)),
        'roc_auc': float(roc_auc_score(y_true, scores)) if pd.Series(y_true).nunique()==2 else np.nan,
        'precision_at_20': precision_at_k(scores, y_true, 20),
        'precision_at_50': precision_at_k(scores, y_true, 50),
        'precision_at_100': precision_at_k(scores, y_true, 100),
        'n_test': int(len(y_true)),
    }

def logistic_pipeline():
    return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                    ('scale', StandardScaler()),
                    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))])

def forest_pipeline():
    return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                    ('model', RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=2,
                        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])

results, predictions = [], []
for split_id, (train_idx, test_idx) in enumerate(gss.split(X, y, groups), start=1):
    train_clients, test_clients = set(groups.iloc[train_idx]), set(groups.iloc[test_idx])
    assert train_clients.isdisjoint(test_clients), 'Client leakage across split.'
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    heldout = eval_df.iloc[test_idx].copy()
    # Existing rule: only pages stale >180 days get priority, ordered by 90-day impressions.
    baseline = (pd.to_numeric(heldout['days_since_last_update'], errors='coerce').fillna(-1).gt(180).astype(int)
                * pd.to_numeric(heldout['impressions_90d'], errors='coerce').fillna(0).clip(lower=0))
    candidates = {'heuristic_stale_traffic': baseline.to_numpy()}
    for name, estimator in [('logistic_regression', logistic_pipeline()), ('random_forest', forest_pipeline())]:
        estimator.fit(X_train, y_train)
        candidates[name] = estimator.predict_proba(X_test)[:,1]
    for model_name, scores in candidates.items():
        results.append({'split':split_id,'model':model_name,**metrics(y_test,scores)})
        ranked = heldout[['content_id','client_id','trend_direction','is_declining_label']].copy()
        ranked['split'] = split_id
        ranked['model'] = model_name
        ranked['priority_score'] = scores
        ranked['rank'] = ranked['priority_score'].rank(method='first',ascending=False).astype(int)
        predictions.append(ranked)

split_results = pd.DataFrame(results)
summary = (split_results.groupby('model')
    .agg(splits=('split','nunique'), mean_average_precision=('average_precision','mean'),
         sd_average_precision=('average_precision','std'), mean_roc_auc=('roc_auc','mean'),
         mean_precision_at_20=('precision_at_20','mean'), mean_precision_at_50=('precision_at_50','mean'),
         sd_precision_at_50=('precision_at_50','std'), mean_precision_at_100=('precision_at_100','mean'),
         mean_test_rows=('n_test','mean'))
    .sort_values('mean_average_precision',ascending=False))
display(summary.round(3))

,splits,mean_average_precision,sd_average_precision,mean_roc_auc,mean_precision_at_20,mean_precision_at_50,sd_precision_at_50,mean_precision_at_100,mean_test_rows
model,,,,,,,,,
random_forest,5,0.610,0.074,0.664,0.66,0.652,0.166,0.652,5878.8
logistic_regression,5,0.570,0.074,0.626,0.65,0.620,0.095,0.568,5878.8
heuristic_stale_traffic,5,0.484,0.095,0.502,0.64,0.580,0.097,0.514,5878.8


## Readout and decision

Compare the models against the unchanged heuristic on average precision and top-k precision, including fold-to-fold spread. Do not infer superiority from one split or from an in-sample queue. The target is the current supplied label, so high performance would not establish that a page can be flagged before decline. Before the later full DuckDB release, verify dated feature and target windows and repeat this same evaluation design on eligible warehouse rows.

In [4]:
OUTPUT_DIR = Path('work/outputs/milestone_3_baseline_model_comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
split_results.to_csv(OUTPUT_DIR/'split_model_metrics.csv', index=False)
summary.to_csv(OUTPUT_DIR/'model_comparison_summary.csv')
prediction_table = pd.concat(predictions, ignore_index=True)
prediction_table.to_csv(OUTPUT_DIR/'heldout_ranked_queues.csv', index=False)
contract = {
    'target':'trend_direction == down; all other recognized labels are non-decline',
    'unit':'one unique content_id within client_id',
    'features':FEATURES,
    'heuristic':'days_since_last_update > 180, ranked by impressions_90d',
    'models':['median_imputed_logistic_regression','median_imputed_random_forest'],
    'missingness':'median imputation plus indicators fitted inside each training split; avg_position zero treated as missing',
    'validation':'5 repeated 75/25 GroupShuffleSplit partitions by client_id; same test rows per candidate within each split',
    'scope':'current-label prioritization only; timestamps unverified; not a future-warning result',
}
(OUTPUT_DIR/'stage_3_contract.json').write_text(json.dumps(contract, indent=2)+'\n', encoding='utf-8')
print('Saved results to', OUTPUT_DIR.resolve())
display(prediction_table.query('split == 1 and model == \"random_forest\"').sort_values('rank').head(20))

Saved results to /content/flyrank-ml-internship-starter/work/outputs/milestone_3_baseline_model_comparison


,content_id,client_id,trend_direction,is_declining_label,split,model,priority_score,rank
15953,content_829c68e60d13,client_f369cb89fc,down,1,1,random_forest,0.672862,1
20872,content_26d48a980581,client_f369cb89fc,up,0,1,random_forest,0.672564,2
17161,content_b84cb2300d9d,client_f369cb89fc,down,1,1,random_forest,0.671297,3
17775,content_d38581402cd5,client_f369cb89fc,down,1,1,random_forest,0.670182,4
18555,content_b08562686d22,client_f369cb89fc,down,1,1,random_forest,0.669989,5
16065,content_fb79d33c4d8a,client_f369cb89fc,up,0,1,random_forest,0.669897,6
14678,content_323d0bdf7077,client_f369cb89fc,down,1,1,random_forest,0.669860,7
15723,content_91c02d2830b8,client_f369cb89fc,stable,0,1,random_forest,0.669747,8
18178,content_515f11d36816,client_f369cb89fc,down,1,1,random_forest,0.669715,9
15998,content_a53d1c4231b3,client_d029fa3a95,stable,0,1,random_forest,0.669579,10
